# 在 VS Code 使用 Colab GPU 訓練 UAV 搜救模型

先在 VS Code 安裝 Google Colab 擴充功能，開啟本 notebook，從右上角選擇 Kernel → Colab → Auto Connect，並登入 Google 帳號。

Colab 是遠端環境，讀不到電腦的 D 槽。請把 stage1_train.py、stage2_train.py、cascade_infer.py 和 sar_stages.zip 放到 Google Drive 的 MyDrive/UAV_Project/（或把 ZIP 放在 MyDrive 根目錄）。ZIP 內應有 data_stage/stage1 與 data_stage/stage2。本 notebook 會把 ZIP 複製到 Colab 暫存空間並解壓，兩階段都從暫存空間讀資料，checkpoint 仍存回 Drive。

Colab 執行階段重置後，/content 暫存空間會清空，需要重新執行 ZIP 複製與解壓儲存格。先依序執行 GPU 檢查、掛載 Drive、專案設定和資料準備；短跑完成後再執行正式訓練。

## 1. 確認目前連到 Colab GPU

先執行下一格確認核心和 CUDA。正常時應顯示 `CUDA available: True` 及 GPU 名稱（例如 Tesla T4）；若顯示 `False`，先重新選擇 Colab GPU 核心，不要開始訓練。

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('目前沒有連上 GPU。請在 VS Code Kernel 選 Colab → Auto Connect，並在 Colab 執行階段設定中選 GPU。')
print('GPU:', torch.cuda.get_device_name(0))

## 2. 掛載 Google Drive

執行下一格授權 Colab 存取 Drive。每次 Colab 執行階段重置後都要重新掛載；模型 checkpoint 和訓練輸出會存回 Drive。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. 設定專案路徑與訓練執行函式

這格會確認 Drive 上有訓練程式和 ZIP，設定輸出資料夾，並建立會即時顯示訓練訊息的 `run_script()`。它不會開始訓練。

In [ ]:
from pathlib import Path
import os, subprocess, sys

PROJECT = Path('/content/drive/MyDrive/UAV_Project')
RUNS = PROJECT / 'runs'
RUNS.mkdir(parents=True, exist_ok=True)

for filename in ['stage1_train.py', 'stage2_train.py', 'cascade_infer.py']:
    if not (PROJECT / filename).is_file():
        raise FileNotFoundError(f'Drive 專案資料夾缺少 {filename}: {PROJECT}')

ZIP_CANDIDATES = [
    PROJECT / 'sar_stages.zip',
    Path('/content/drive/MyDrive/sar_stages.zip'),
]
ZIP_DRIVE = next((path for path in ZIP_CANDIDATES if path.is_file()), None)
if ZIP_DRIVE is None:
    raise FileNotFoundError(
        '找不到 sar_stages.zip，請放到 MyDrive/UAV_Project/ 或 MyDrive 根目錄。'
    )

print('專案程式與 ZIP 檔案已找到')
print('PROJECT =', PROJECT)
print('ZIP_DRIVE =', ZIP_DRIVE)
os.chdir(PROJECT)

def run_script(*args):
    command = [sys.executable, '-u', *map(str, args)]
    print('執行：', ' '.join(command), flush=True)

    process = subprocess.Popen(
        command,
        cwd=PROJECT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end='', flush=True)

    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)

## 將 sar_stages.zip 複製並解壓到 Colab 暫存空間

這格會從 Drive 複製單一 ZIP，再解壓到 /content。程式會自動尋找同時包含 stage1 和 stage2 的 data_stage 根目錄，並檢查兩階段需要的資料夾。若目前執行階段已經有完整解壓資料，會直接沿用。

In [ ]:
import zipfile

LOCAL_ZIP = Path('/content/sar_stages.zip')
LOCAL_ROOT = Path('/content/data')
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)

def find_dataset_root(base):
    for stage1_dir in base.rglob('stage1'):
        if stage1_dir.is_dir():
            root = stage1_dir.parent
            if (root / 'stage2').is_dir():
                return root
    return None

DATA = find_dataset_root(LOCAL_ROOT)

if DATA is None:
    total_bytes = ZIP_DRIVE.stat().st_size
    print(f'開始從 Drive 複製 ZIP：{total_bytes / (1024**3):.2f} GB', flush=True)
    copied = 0
    last_reported = 0
    with ZIP_DRIVE.open('rb') as source, LOCAL_ZIP.open('wb') as target:
        while True:
            chunk = source.read(8 * 1024 * 1024)
            if not chunk:
                break
            target.write(chunk)
            copied += len(chunk)
            if copied - last_reported >= 64 * 1024 * 1024 or copied == total_bytes:
                percent = 100 * copied / max(total_bytes, 1)
                print(f'ZIP 複製進度：{percent:.1f}% ({copied / (1024**3):.2f} GB)', flush=True)
                last_reported = copied

    print('ZIP 複製完成，開始解壓縮...', flush=True)
    with zipfile.ZipFile(LOCAL_ZIP) as archive:
        files = [item for item in archive.infolist() if not item.is_dir()]
        for i, item in enumerate(files, start=1):
            archive.extract(item, LOCAL_ROOT)
            if i % 1000 == 0 or i == len(files):
                print(f'解壓進度：{i}/{len(files)} 個檔案', flush=True)

    DATA = find_dataset_root(LOCAL_ROOT)

if DATA is None:
    raise FileNotFoundError(
        '解壓後找不到同時包含 stage1/ 與 stage2/ 的資料根目錄；'
        '請確認 ZIP 內有 data_stage/stage1 與 data_stage/stage2。'
    )

required_folders = [
    DATA / 'stage1/train/images',
    DATA / 'stage1/train/grid',
    DATA / 'stage1/test/images',
    DATA / 'stage1/test/grid',
    DATA / 'stage2/train/person',
    DATA / 'stage2/train/not_person',
    DATA / 'stage2/test/person',
    DATA / 'stage2/test/not_person',
]
missing = [str(folder) for folder in required_folders if not folder.is_dir()]
if missing:
    raise FileNotFoundError('資料集缺少必要資料夾：' + ', '.join(missing))

print('資料集根目錄：', DATA)
print('Stage 1 與 Stage 2 資料夾檢查完成')

## Stage 1 短跑

這格只訓練 2 個 epoch，用來確認資料、程式和 GPU 都能正常工作。輸出存到 Drive 的 `runs/stage1/colab_smoke.pt`，不會覆蓋正式模型。

In [ ]:
run_script(PROJECT/'stage1_train.py', '--data', DATA, '--epochs', 2, '--batch-size', 4, '--workers', 2, '--out', RUNS/'stage1/colab_smoke.pt')

## Stage 1 正式訓練（確認短跑完成後才執行）

60 epochs；最佳 checkpoint 存到 `runs/stage1/best_enhanced.pt`。

In [ ]:
run_script(PROJECT/'stage1_train.py', '--data', DATA, '--epochs', 60, '--batch-size', 8, '--workers', 2, '--out', RUNS/'stage1/best_enhanced.pt')

## Stage 2 短跑

確認 Stage 2 資料夾都有 `person` 和 `not_person` 影像後，執行 1 epoch 短跑。輸出不會覆蓋正式模型。

In [ ]:
run_script(PROJECT/'stage2_train.py', '--data', DATA, '--epochs', 1, '--batch-size', 16, '--workers', 2, '--out', RUNS/'stage2/colab_smoke.pt')

## Stage 2 正式訓練（確認短跑完成後才執行）

30 epochs；最佳 checkpoint 存到 `runs/stage2/best_stage2.pt`。

In [ ]:
run_script(PROJECT/'stage2_train.py', '--data', DATA, '--epochs', 30, '--batch-size', 32, '--workers', 2, '--out', RUNS/'stage2/best_stage2.pt')

## 可選：串接推論

如果也把 `sar_test_raw` 放到 Drive 專案資料夾，可用一張原始高解析度影像測試 Stage 1 + Stage 2。請先完成兩個正式 checkpoint。

In [ ]:
TEST_IMAGE = PROJECT/'sar_test_raw/dataset_det/test/rgb_0000.jpg'
if not TEST_IMAGE.is_file():
    raise FileNotFoundError(f'找不到測試影像：{TEST_IMAGE}')
run_script(PROJECT/'cascade_infer.py', '--image', TEST_IMAGE, '--stage1', RUNS/'stage1/best_enhanced.pt', '--stage2', RUNS/'stage2/best_stage2.pt', '--output', RUNS/'cascade/result.jpg', '--save-crops')